# Arricchimento Preamboli via EUR-Lex HTML

Questo notebook recupera il **preambolo** (considerando) di ciascun atto normativo del grafo focale tramite `eurlex.get_html_by_celex_id()`.

## Perché il preambolo

Il preambolo è la parte più informativa per la classificazione semantica nei livelli normativi:
- Dichiara esplicitamente la **base giuridica** (quale articolo del TFUE)
- Indica **chi ha adottato** l'atto e con quale procedura
- Enuncia gli **obiettivi** e il contesto normativo
- È strutturalmente standardizzato in tutti gli atti UE

A differenza del titolo, il preambolo permette di distinguere atti formalmente simili ma funzionalmente diversi — es. un Regolamento quadro (G2) da un Regolamento tecnico-operativo (G3).

## Struttura HTML

Il preambolo si trova nel primo tag `eli-subdivision` della pagina EUR-Lex, dal primo carattere fino al marker `HAVE ADOPTED THIS REGULATION:` (o equivalente per Direttive/Decisioni).

## Parametri
- **Delay**: 0.5s tra richieste
- **Checkpoint**: ogni 50 nodi
- **Input**: `data/output/golden_power/gephi_nodes_focal.csv`
- **Output**: `data/output/golden_power/gephi_nodes_focal_preambles.csv`

## 0. Setup

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import eurlex
import time
import os
import sys

sys.path.append('..')
from config_golden_power import MATERIA_NAME

output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file      = os.path.join(output_path, 'gephi_nodes_focal.csv')
output_file     = os.path.join(output_path, 'gephi_nodes_focal_preambles.csv')
checkpoint_file = os.path.join(output_path, 'preambles_checkpoint.csv')

DELAY_SECONDS    = 0.5
CHECKPOINT_EVERY = 50
MAX_PREAMBLE_CHARS = 3000  # tronca preamboli molto lunghi per efficienza LLM

# Marker che segnalano la fine del preambolo e l'inizio dell'articolato
PREAMBLE_END_MARKERS = [
    'HAVE ADOPTED THIS REGULATION:',
    'HAS ADOPTED THIS REGULATION:',
    'HAVE ADOPTED THIS DIRECTIVE:',
    'HAS ADOPTED THIS DIRECTIVE:',
    'HAVE ADOPTED THIS DECISION:',
    'HAS ADOPTED THIS DECISION:',
    'HAVE ADOPTED THIS FRAMEWORK DECISION:',
    'HEREBY DECIDES:',
    'HAS DECIDED AS FOLLOWS:',
    'HEREBY RECOMMENDS:',
    'IS OF THE OPINION THAT:',
]

print(f"Input:      {input_file}")
print(f"Output:     {output_file}")
print(f"Checkpoint: {checkpoint_file}")

Input:      ..\data\output\golden_power\gephi_nodes_focal.csv
Output:     ..\data\output\golden_power\gephi_nodes_focal_preambles.csv
Checkpoint: ..\data\output\golden_power\preambles_checkpoint.csv


## 1. Caricamento Nodi e Gestione Checkpoint

In [2]:
nodes = pd.read_csv(input_file)
print(f"Nodi totali nel grafo focale: {len(nodes)}")

if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi gia processati")
    print(f"Nodi rimanenti:     {len(nodes) - len(already_done)}")
else:
    checkpoint   = pd.DataFrame(columns=['Id', 'Label', 'preamble', 'preamble_status', 'preamble_length'])
    already_done = set()
    print("Nessun checkpoint trovato, si parte da zero")

nodes_todo = nodes[~nodes['Id'].isin(already_done)].copy()
print(f"Da processare ora: {len(nodes_todo)}")

Nodi totali nel grafo focale: 4904
Checkpoint trovato: 2766 nodi gia processati
Nodi rimanenti:     2138
Da processare ora: 2432


## 2. Funzione di Estrazione Preambolo

Il preambolo è nel primo tag `eli-subdivision` della pagina HTML. Viene troncato al marker che segnala l'inizio dell'articolato (`HAVE ADOPTED THIS REGULATION:` ecc.) e poi troncato a `MAX_PREAMBLE_CHARS` caratteri per contenere il contesto LLM.

In [3]:
def get_preamble_from_eurlex(celex):
    """
    Recupera il preambolo di un atto EUR-Lex dal codice CELEX.

    Restituisce (preamble, status) dove status e:
      'ok'        - preambolo recuperato con successo
      'not_found' - pagina non trovata o struttura HTML diversa
      'error'     - errore generico
    """
    if pd.isna(celex) or celex == '':
        return None, 'not_found'

    try:
        html = eurlex.get_html_by_celex_id(celex, language = 'en')
        if not html:
            return None, 'not_found'

        soup         = BeautifulSoup(html, 'html.parser')
        subdivisions = soup.find_all(class_='eli-subdivision')

        if not subdivisions:
            return None, 'not_found'

        testo = subdivisions[0].get_text(strip=True)

        # Taglia all'inizio dell'articolato
        for marker in PREAMBLE_END_MARKERS:
            if marker in testo:
                testo = testo[:testo.index(marker)]
                break

        testo = testo.strip()
        if not testo:
            return None, 'not_found'

        # Tronca per efficienza LLM
        testo = testo[:MAX_PREAMBLE_CHARS]

        return testo, 'ok'

    except Exception:
        return None, 'error'


# Test
print("Test su 32019R0452 (FDI Screening)...")
preamble, status = get_preamble_from_eurlex('32019R0452')
print(f"  Status:    {status}")
print(f"  Lunghezza: {len(preamble)} caratteri")
print(f"  Inizio:    {preamble[:150]}")
print()
print("Test su 32008L0114 (Dir. Infrastrutture critiche)...")
preamble2, status2 = get_preamble_from_eurlex('32008L0114')
print(f"  Status:    {status2}")
print(f"  Lunghezza: {len(preamble2)} caratteri")

Test su 32019R0452 (FDI Screening)...
  Status:    ok
  Lunghezza: 3000 caratteri
  Inizio:    THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty on the Functioning of the European Union, and in particular 

Test su 32008L0114 (Dir. Infrastrutture critiche)...
  Status:    ok
  Lunghezza: 3000 caratteri


## 3. Fetch con Checkpoint

Con 2.766 nodi e 0.5s di delay il tempo stimato e circa **25 minuti**. Se viene interrotto, riesegui questa cella: ripartira dal checkpoint automaticamente.

In [4]:
results = []
total   = len(nodes_todo)
n_ok    = 0
n_err   = 0

print(f"Inizio fetch: {total} nodi")
print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    celex   = row['Label']
    node_id = row['Id']

    preamble, status = get_preamble_from_eurlex(celex)

    if status == 'ok':
        n_ok += 1
    else:
        n_err += 1

    results.append({
        'Id':               node_id,
        'Label':            celex,
        'preamble':         preamble,
        'preamble_status':  status,
        'preamble_length':  len(preamble) if preamble else 0,
    })

    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct = (i + 1) / total * 100
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {n_ok}  errori: {n_err}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch              = pd.DataFrame(results)
        checkpoint_updated = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
        checkpoint_updated.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint salvato ({len(checkpoint_updated)} nodi totali)")

    time.sleep(DELAY_SECONDS)

# Checkpoint finale
batch            = pd.DataFrame(results)
checkpoint_final = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
checkpoint_final.to_csv(checkpoint_file, index=False)

print(f"\nFetch completato.")
print(f"  OK:        {(checkpoint_final['preamble_status'] == 'ok').sum()}")
print(f"  Not found: {(checkpoint_final['preamble_status'] == 'not_found').sum()}")
print(f"  Errori:    {(checkpoint_final['preamble_status'] == 'error').sum()}")

Inizio fetch: 2432 nodi
Tempo stimato: ~20 minuti

  [  10/2432]   0.4%  ok: 2  errori: 8
  [  20/2432]   0.8%  ok: 10  errori: 10
  [  30/2432]   1.2%  ok: 20  errori: 10
  [  40/2432]   1.6%  ok: 30  errori: 10
  [  50/2432]   2.1%  ok: 40  errori: 10
  --> Checkpoint salvato (2816 nodi totali)
  [  60/2432]   2.5%  ok: 46  errori: 14
  [  70/2432]   2.9%  ok: 50  errori: 20
  [  80/2432]   3.3%  ok: 60  errori: 20
  [  90/2432]   3.7%  ok: 70  errori: 20
  [ 100/2432]   4.1%  ok: 79  errori: 21
  --> Checkpoint salvato (2866 nodi totali)
  [ 110/2432]   4.5%  ok: 89  errori: 21
  [ 120/2432]   4.9%  ok: 93  errori: 27
  [ 130/2432]   5.3%  ok: 103  errori: 27
  [ 140/2432]   5.8%  ok: 113  errori: 27
  [ 150/2432]   6.2%  ok: 123  errori: 27
  --> Checkpoint salvato (2916 nodi totali)
  [ 160/2432]   6.6%  ok: 133  errori: 27
  [ 170/2432]   7.0%  ok: 142  errori: 28
  [ 180/2432]   7.4%  ok: 148  errori: 32
  [ 190/2432]   7.8%  ok: 157  errori: 33
  [ 200/2432]   8.2%  ok: 167  er

## 4. Export

In [8]:
preambles_df   = pd.read_csv(checkpoint_file)[['Id', 'preamble', 'preamble_status', 'preamble_length']]
nodes_enriched = nodes.merge(preambles_df, on='Id', how='left')
nodes_enriched.to_csv(output_file, index=False)

print(f"File salvato: {output_file}")
print(f"  Nodi totali:      {len(nodes_enriched)}")
print(f"  Con preambolo:    {nodes_enriched['preamble'].notna().sum()} ({nodes_enriched['preamble'].notna().sum()/len(nodes_enriched)*100:.1f}%)")
print(f"  Senza preambolo:  {nodes_enriched['preamble'].isna().sum()}")
print()
print("Distribuzione lunghezza preambolo (caratteri):")
print(nodes_enriched['preamble_length'].describe().round(0).to_string())

File salvato: ..\data\output\golden_power\gephi_nodes_focal_preambles.csv
  Nodi totali:      4904
  Con preambolo:    3177 (64.8%)
  Senza preambolo:  1727

Distribuzione lunghezza preambolo (caratteri):
count    4904.0
mean     1716.0
std      1375.0
min         0.0
25%         0.0
50%      2522.0
75%      3000.0
max      3000.0


## 5. Diagnostica

I nodi senza preambolo sono tipicamente articoli di trattati, atti molto vecchi, o atti con struttura HTML non standard. Per la classificazione LLM verranno saltati o gestiti con fallback sul titolo.

In [6]:
print("Distribuzione status:")
print(nodes_enriched['preamble_status'].value_counts().to_string())
print()

no_preamble = nodes_enriched[nodes_enriched['preamble'].isna()]
if len(no_preamble) > 0:
    print(f"Nodi senza preambolo: {len(no_preamble)}")
    print("Per tipo:")
    print(no_preamble['LegalType'].value_counts().to_string())
    print()
    print("Per decade:")
    print(no_preamble['Decade'].value_counts().sort_index().to_string())
else:
    print("Tutti i nodi hanno un preambolo.")

Distribuzione status:
preamble_status
ok           3177
not_found    1727

Nodi senza preambolo: 1727
Per tipo:
LegalType
Treaty               485
Decision             403
Case_Law             221
Directive            220
Regulation           184
Legislative_Act      135
Recommendation        62
Complementary_Act     16
Guidelines             1

Per decade:
Decade
1950.0     52
1960.0     25
1970.0     75
1980.0    150
1990.0    558
2000.0    441
2010.0    380
2020.0     45
